# Agents Demo: Modeling, Evaluation, and OpenAI Review

This notebook shows how to orchestrate a full Model Zoo workflow using the `ModelingAgent`, `EvaluationAgent`, and the optional `OpenAIReviewerAgent`.

## Prerequisites
- A Model Zoo API server running locally at `http://localhost:8000`.
- Required packages installed via `pip install -e .` (from the repo root).
- Optional: set `OPENAI_API_KEY` (or pass `api_key` explicitly) to enable OpenAI-based review.

If no API key is available, the reviewer agent falls back to a heuristic summary.

In [ ]:
import numpy as np
from agents import ModelingAgent, EvaluationAgent, OpenAIReviewerAgent


In [ ]:
config_payload = {
    'model': {
        'id': 'demo_logit_agent_v1',
        'name': 'Agent Logistic Demo',
        'version': '1.0.0',
        'kind': 'library',
        'binding': {
            'module': 'custom_logit_model',
            'callable': 'train_and_classify'
        },
        'inputs': ['feature_matrix', 'labels']
    },
    'dependencies': [
        'custom-logit-model==0.1.0',
        'scikit-learn>=1.4',
        'numpy>=1.24'
    ],
    'hyperparameters': {
        'schema': [
            {'name': 'solver', 'dtype': 'string', 'default': 'lbfgs'},
            {'name': 'max_iter', 'dtype': 'int', 'default': 200}
        ],
        'values': [
            {'model_id': 'demo_logit_agent_v1', 'param': 'solver', 'value': 'lbfgs', 'env': 'prod'},
            {'model_id': 'demo_logit_agent_v1', 'param': 'max_iter', 'value': '250', 'env': 'prod'}
        ]
    },
    'context': {'timeout_s': 180}
}


In [ ]:
rng = np.random.default_rng(17)
X = rng.normal(size=(200, 4))
true_coef = np.array([1.5, -2.0, 0.7, 0.5])
logits = X @ true_coef
probabilities = 1 / (1 + np.exp(-logits))
y = (probabilities > 0.5).astype(int)

data_sources = {
    'feature_matrix': X.tolist(),
    'labels': y.tolist(),
}


In [ ]:
modeling_agent = ModelingAgent(run_timeout=240)
evaluation_agent = EvaluationAgent(
    metric_thresholds={'accuracy': 0.7},
    required_artifacts=['probabilities']
)
try:
    reviewer_agent = OpenAIReviewerAgent()
    print('OpenAI reviewer enabled.')
except Exception as exc:
    print(f'Reviewer unavailable: {exc}')
    reviewer_agent = OpenAIReviewerAgent(enabled=False)


In [ ]:
try:
    result = modeling_agent.execute_workflow(
        config_payload=config_payload,
        data_sources=data_sources,
        env='prod',
        params={'solver': 'lbfgs'},
        context={'timeout_s': 180},
        evaluation_agent=evaluation_agent,
        reviewer_agent=reviewer_agent if reviewer_agent.enabled else None,
    )
    result
except Exception as exc:
    print(f'Workflow failed: {exc}')
